# Demonstration of YOLO inference

## 0. Preparation

### Python & Virtual Environment

Before running the script, ensure that Python is installed on your system. The script has been tested with Python 3.11.7 and pip 23.2.1. It is recommended to use a virtual environment to manage dependencies and avoid conflicts with other Python packages on your system.

### Creating a Virtual Environment

To create and activate a virtual environment, follow these steps:

1. **Create the virtual environment**:  
   In the terminal, navigate to the project directory and run:
   ```bash
   python -m venv bench_reitoria_env
   ```
   This will create a directory named `bench_reitoria_env` in your project directory, which will contain the isolated Python environment.

2. **Activate the virtual environment**:
   - **macOS/Linux**:
     ```bash
     source bench_reitoria_env/bin/activate
     ```

   Once activated, the terminal prompt should change to indicate that the virtual environment is active, e.g., `(bench_reitoria_env)`.

3. **Install required libraries**:  
   With the virtual environment active, run the following command to install all necessary dependencies:
   ```bash
   pip install -r requirements.txt
   ```

   This will install the required libraries

4. **Deactivate the virtual environment**:  
   After you're done working, you can deactivate the virtual environment by running:
   ```bash
   deactivate
   ```



to Select the interpreter in VS Code :

`Ctrl + Shift + P`

`Python: Select Interpreter`

Choose your venv (it will show something like): 
`./venv/bin/python3.11.2`


## 1. Import required modules:

In [ ]:
import os

os.getcwd()
import re
from datetime import datetime, timedelta
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
from ultralytics import YOLO
import time
import cv2
import numpy as np
import json
import argparse
import imageio.v3 as iio
from utils_yolo import *
import psutil  # For system resource monitoring
import csv
import glob
from IPython.display import Image, display
from PIL import Image
import pandas as pd


## 2. Set variables

In [ ]:
model_name = 'yolo11m'
output_path = f'demo_output_{model_name}'
input_path = "bench_reitoria_challenge"
savefigs = 'debug'
patching = False
labels_csv = f'{input_path}/labels.csv'
run_at_cpu = False


## 2. Instantiate a YOLO detection model

In [ ]:

model = YOLO(f"{model_name}.pt")

if run_at_cpu:
    model = model.to('cpu')

## 3. Inspect paths

In [ ]:

if os.path.exists(output_path):
    print(f"Output path already exists: {output_path}")
else:
    os.makedirs(output_path)
    print(f"Output path created: {output_path}")

# Prepare CSV output
csv_file_path = os.path.join(output_path, 'parking_results.csv')
csv_headers = [
'image_name', 'predicted_cars', 'processing_time',
'trimmed_processing_time',
'avg_normal_time', 'avg_trimmed_time',
'cpu_usage', 'memory_used', 'swap_used', 'patching', 'timestamp'
]

with open(csv_file_path, 'w', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_headers)
    writer.writeheader()
print("input path :",input_path)


image_list = list_images(input_path)
print(f'images list: {image_list}')

labels_csv = f'{input_path}/labels.csv'
labels_csv

labels_df = pd.read_csv(labels_csv)
print('labels file:')
labels_df


## 4. Run the inference on all files from input directory

In [ ]:
df = None
processing_times = []
if image_list:
    
    for image_file in image_list:
        print(f"Processing: {image_file}")
        image_path = os.path.join(input_path, image_file)
        img_object = Image.open(f'{input_path}/{image_file}')
        image_timestamp = extract_timestamp(image_path, mode='filename_reitoria')
        # img_ =  padronize_filename(image_file,image_timestamp)
        img_ = image_file
        print(f"img_: {img_}\n\n\nimg_object: {img_object}")

        start_time = time.time()
        start_metrics = get_system_metrics()

        if patching:
            print('Patching enabled, performing 6 blocks processing \n\n')
            cars, img1, preench1, image1_annotations = perform_inference_blocks(img_object, model, img_,df, output_path, save=savefigs)
        else:
            print('Patching disabled, performing regular processing \n\n')
            cars, img1, preench1, image1_annotations = perform_inference(img_object, model, img_, output_path, save=savefigs)
        patching_flag = patching

        # Calculate processing metrics
        processing_time = time.time() - start_time
        end_metrics = get_system_metrics()
        # Keep a history of all processing times
        processing_times.append(processing_time)

        # Compute trimmed processing time (per-row)
        trimmed_time = None
        if len(processing_times) > 20:
            trimmed_slice = processing_times[10:-10]
            if len(trimmed_slice) > 0:
                trimmed_time = sum(trimmed_slice) / len(trimmed_slice)

        # Compute running averages
        avg_normal_time = sum(processing_times) / len(processing_times)

        avg_trimmed_time = None
        if len(processing_times) > 20:
            trimmed_slice = processing_times[10:-10]
            if len(trimmed_slice) > 0:
                avg_trimmed_time = sum(trimmed_slice) / len(trimmed_slice)

        # Average the metrics (start and end)
        avg_metrics = {
            'cpu': (start_metrics['cpu'] + end_metrics['cpu']) / 2,
            'memory': (start_metrics['memory'] + end_metrics['memory']) / 2,
            'swap': (start_metrics['swap'] + end_metrics['swap']) / 2
        }

        # Write to CSV
        with open(csv_file_path, 'a', newline='') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=csv_headers)
            writer.writerow({
                'image_name': image_file,
                'predicted_cars': cars,
                'processing_time': processing_time,
                'cpu_usage': avg_metrics['cpu'],
                'memory_used': avg_metrics['memory'],
                'swap_used': avg_metrics['swap'],
                'patching': patching_flag if ('cam_reitoria_1' in img_ or 'cam_reitoria_4' in img_) else False,
                'timestamp': image_timestamp,
                'trimmed_processing_time': trimmed_time,
                'avg_normal_time': avg_normal_time,
                'avg_trimmed_time': avg_trimmed_time,


            })

        print(f"Processed {image_file} - Cars: {cars} - Time: {processing_time:.2f}s")

## 5. Evaluate results

In [ ]:

# Load the CSV file created by your loop
# (Ensure csv_file_path variable is still in memory, or replace with the actual filename string)
df_results = pd.read_csv(csv_file_path)

# 1. See the most recent entries
print("Latest processed images:")
display(df_results.tail())

# 2. Check statistics (e.g., avg processing time, avg cars found)
print("\nStatistics:")
display(df_results[['predicted_cars', 'processing_time', 'cpu_usage']].describe())

# 3. Plot processing time over time to check for spikes
df_results['processing_time'].plot(title="Processing Time per Image", figsize=(10, 4))

Note that max_spots is a placeholder and is mostly used for IC parking to calculate accuracy metrics.

In [ ]:
if patching:
    suffix='patch'
else:
    suffix='nopatch'

! python3 compute_metrics.py \
    --model "{model_name}_{suffix}" \
    --parking_metrics "{output_path}/parking_results.csv" \
    --labels "{labels_csv}" \
    --output_dir "{output_path}/" \
    --max_spots {15}

In [ ]:
print(f"Contents of {output_path}:")
print(os.listdir(output_path))

In [ ]:

# output_dir = 'demo_output_yolo11m'
summary_path = os.path.join(output_path, f'summary_metrics_{model_name}_{suffix}.csv')
# output_path
df = pd.read_csv(summary_path)

# List of values to exclude
exclude = ['Average accuracy', 'Balanced accuracy', 'Average precision', 'Average recall', 'Average F1 score']
df_filtered = df[~df.iloc[:, 0].isin(exclude)]

display(df_filtered)

In [ ]:

show_images = 1
# Get all result images
result_images = glob.glob(os.path.join(output_path, 'results_*.jpg'))

# Display the first image
for img_path in result_images[:show_images]:
    print(f"Displaying: {os.path.basename(img_path)}")
    img = Image.open(img_path)
    display(img)

## Inference with Patching

preparation

In [ ]:
patching = True
if patching:
    suffix='patch'
else:
    suffix='nopatch'

model_name = 'yolo11m'
output_path = f'demo_output_{model_name}_{suffix}'
input_path = "bench_reitoria_challenge"
savefigs = 'debug'

labels_csv = f'{input_path}/labels.csv'
run_at_cpu = False

model = YOLO(f"{model_name}.pt")
if run_at_cpu:
    model = model.to('cpu')

if os.path.exists(output_path):
    print(f"Output path already exists: {output_path}")
else:
    os.makedirs(output_path)
    print(f"Output path created: {output_path}")

# Prepare CSV output
csv_file_path = os.path.join(output_path, 'parking_results.csv')
csv_headers = [
'image_name', 'predicted_cars', 'processing_time',
'trimmed_processing_time',
'avg_normal_time', 'avg_trimmed_time',
'cpu_usage', 'memory_used', 'swap_used', 'patching', 'timestamp'
]

with open(csv_file_path, 'w', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_headers)
    writer.writeheader()
print("input path :",input_path)


image_list = list_images(input_path)
print(f'images list: {image_list}')

labels_csv = f'{input_path}/labels.csv'
labels_csv

labels_df = pd.read_csv(labels_csv)
print('labels file:')
labels_df

df = None
processing_times = []
if image_list:
    
    for image_file in image_list:
        print(f"Processing: {image_file}")
        image_path = os.path.join(input_path, image_file)
        img_object = Image.open(f'{input_path}/{image_file}')
        image_timestamp = extract_timestamp(image_path, mode='filename_reitoria')
        # img_ =  padronize_filename(image_file,image_timestamp)
        img_ = image_file
        print(f"img_: {img_}\n\n\nimg_object: {img_object}")

        start_time = time.time()
        start_metrics = get_system_metrics()

        if patching:
            print('Patching enabled, performing 6 blocks processing \n\n')
            cars, img1, preench1, image1_annotations = perform_inference_blocks(img_object, model, img_,df, output_path, save=savefigs)
        else:
            print('Patching disabled, performing regular processing \n\n')
            cars, img1, preench1, image1_annotations = perform_inference(img_object, model, img_, output_path, save=savefigs)
        patching_flag = patching

        # Calculate processing metrics
        processing_time = time.time() - start_time
        end_metrics = get_system_metrics()
        # Keep a history of all processing times
        processing_times.append(processing_time)

        # Compute trimmed processing time (per-row)
        trimmed_time = None
        if len(processing_times) > 20:
            trimmed_slice = processing_times[10:-10]
            if len(trimmed_slice) > 0:
                trimmed_time = sum(trimmed_slice) / len(trimmed_slice)

        # Compute running averages
        avg_normal_time = sum(processing_times) / len(processing_times)

        avg_trimmed_time = None
        if len(processing_times) > 20:
            trimmed_slice = processing_times[10:-10]
            if len(trimmed_slice) > 0:
                avg_trimmed_time = sum(trimmed_slice) / len(trimmed_slice)

        # Average the metrics (start and end)
        avg_metrics = {
            'cpu': (start_metrics['cpu'] + end_metrics['cpu']) / 2,
            'memory': (start_metrics['memory'] + end_metrics['memory']) / 2,
            'swap': (start_metrics['swap'] + end_metrics['swap']) / 2
        }

        # Write to CSV
        with open(csv_file_path, 'a', newline='') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=csv_headers)
            writer.writerow({
                'image_name': image_file,
                'predicted_cars': cars,
                'processing_time': processing_time,
                'cpu_usage': avg_metrics['cpu'],
                'memory_used': avg_metrics['memory'],
                'swap_used': avg_metrics['swap'],
                'patching': patching_flag if ('cam_reitoria_1' in img_ or 'cam_reitoria_4' in img_) else False,
                'timestamp': image_timestamp,
                'trimmed_processing_time': trimmed_time,
                'avg_normal_time': avg_normal_time,
                'avg_trimmed_time': avg_trimmed_time,


            })

        print(f"Processed {image_file} - Cars: {cars} - Time: {processing_time:.2f}s")


In [ ]:

# Load the CSV file created by your loop
# (Ensure csv_file_path variable is still in memory, or replace with the actual filename string)
df_results = pd.read_csv(csv_file_path)

# 1. See the most recent entries
print("Latest processed images:")
display(df_results.tail())

# 2. Check statistics (e.g., avg processing time, avg cars found)
print("\nStatistics:")
display(df_results[['predicted_cars', 'processing_time', 'cpu_usage']].describe())

# 3. Plot processing time over time to check for spikes
df_results['processing_time'].plot(title="Processing Time per Image", figsize=(10, 4))

In [ ]:
! python3 compute_metrics.py \
    --model "{model_name}_{suffix}" \
    --parking_metrics "{output_path}/parking_results.csv" \
    --labels "{labels_csv}" \
    --output_dir "{output_path}/" \
    --max_spots {15}

In [ ]:
print(f"Contents of {output_path}:")
print(os.listdir(output_path))


# output_dir = 'demo_output_yolo11m'
summary_path = os.path.join(output_path, f'summary_metrics_{model_name}_{suffix}.csv')
# output_path
df = pd.read_csv(summary_path)

# List of values to exclude
exclude = ['Average accuracy', 'Balanced accuracy', 'Average precision', 'Average recall', 'Average F1 score']
df_filtered = df[~df.iloc[:, 0].isin(exclude)]

display(df_filtered)


show_images = 1
# Get all result images
result_images = glob.glob(os.path.join(output_path, 'results*.jpg'))

# Display the first image
for img_path in result_images[:show_images]:
    print(f"Displaying: {os.path.basename(img_path)}")
    img = Image.open(img_path)
    display(img)

## Inference with SAHI

Import libraries

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["YOLO_DEVICE"] = "cpu"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

import torch
torch.cuda.is_available = lambda : False
torch.cuda.device_count = lambda : 0
# ===============================
from sahi.utils.file import download_from_url
from sahi.utils.ultralytics import download_yolo11n_model
from sahi import AutoDetectionModel
from sahi.predict import get_prediction
from PIL import Image
from sahi.predict import get_sliced_prediction
from tqdm import tqdm
import itertools

functions

In [ ]:

def plot_side_by_side(
    image1,
    image2,
    output_dir,
    img_name,
    vehicle_count_before,
    vehicle_count_after,
    config_name,
    save_figures=True,
):
    if not save_figures:
        return

    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    ax[0].imshow(image1)
    ax[0].set_title(f"No SAHI Image: {vehicle_count_before} cars")
    ax[0].axis("off")

    ax[1].imshow(image2)
    ax[1].set_title(f"SAHI {config_name}: {vehicle_count_after} cars")
    ax[1].axis("off")

    plt.tight_layout()
    plt.savefig(f"{output_dir}/comparison_{img_name}_{config_name}.png")
    plt.close()


def calculate_overlap_params(slice_width, slice_height, overlap_ratio):
    """Calculate x_overlap and y_overlap based on slice dimensions and overlap ratio"""
    x_overlap = int(overlap_ratio * slice_width)
    y_overlap = int(overlap_ratio * slice_height)
    return x_overlap, y_overlap



def evaluate_sahi_config(
    detection_model,
    input_image,
    config,
    output_dir,
    img_name,
    exclude_classes_by_id,
    vehicle_class_ids,
    save_figures=True,
):
    """Evaluate a single SAHI configuration"""
    start_time = time.time()
    start_metrics = get_system_metrics()

    # Calculate overlaps based on ratio (for recording only)
    x_overlap, y_overlap = calculate_overlap_params(
        config["slice_width"], config["slice_height"], config["overlap_ratio"]
    )

    # MODIFIED: First, let SAHI calculate the actual slice parameters
    from sahi.slicing import get_slice_bboxes
    from sahi.utils.cv import read_image_as_pil
    
    # Load image to get dimensions
    image_pil = read_image_as_pil(input_image)
    w, h = image_pil.size
    
    # Get SAHI's actual calculation with auto_slice_resolution=True
    slice_bboxes = get_slice_bboxes(
        image_height=h,
        image_width=w,
        slice_height=None,  # Let SAHI decide
        slice_width=None,   # Let SAHI decide
        overlap_height_ratio=0.2,  # Default
        overlap_width_ratio=0.2,   # Default
        auto_slice_resolution=True,
    )
    
    # Calculate what SAHI actually used
    if slice_bboxes:
        first_bbox = slice_bboxes[0]
        actual_slice_width = first_bbox[2] - first_bbox[0]
        actual_slice_height = first_bbox[3] - first_bbox[1]
        actual_num_slices = len(slice_bboxes)
        print(f"SAHI auto calculation: {actual_slice_width}x{actual_slice_height}, {actual_num_slices} slices")
    else:
        actual_slice_width = w
        actual_slice_height = h
        actual_num_slices = 1

    # Run SAHI prediction with auto_slice_resolution
    result = get_sliced_prediction(
        input_image,
        detection_model,
        slice_height=None,  # Let SAHI decide
        slice_width=None,   # Let SAHI decide
        overlap_height_ratio=0.2,  # Default
        overlap_width_ratio=0.2,   # Default
        postprocess_type=config["postprocess_type"],
        postprocess_match_metric=config["postprocess_match_metric"],
        postprocess_match_threshold=config["postprocess_match_threshold"],
        verbose=0,
        auto_slice_resolution=True,  # This is key!
        exclude_classes_by_id=exclude_classes_by_id,
    )

    # Calculate metrics
    processing_time = time.time() - start_time
    end_metrics = get_system_metrics()
    avg_metrics = {
        "cpu": (start_metrics["cpu"] + end_metrics["cpu"]) / 2,
        "memory": (start_metrics["memory"] + end_metrics["memory"]) / 2,
        "swap": (start_metrics["swap"] + end_metrics["swap"]) / 2,
    }

    vehicle_count_after = sum(
        1
        for pred in result.object_prediction_list
        if pred.category.id in vehicle_class_ids
    )

    # Save visuals
    config_name = f"auto_{actual_slice_width}x{actual_slice_height}_ov0.2_{config['postprocess_type']}_{config['postprocess_match_metric']}"

    if save_figures:
        result.export_visuals(
            export_dir=output_dir,
            rect_th=3,
            hide_labels=False,
            hide_conf=True,
            file_name=f"result_SAHI_{config_name}_{img_name}",
        )

    # MODIFIED: Return both config values and actual values used by SAHI
    return {
        "image_name": img_name,
        "config_name": config_name,
        "predicted_cars": vehicle_count_after,
        "processing_time": processing_time,
        "cpu_usage": avg_metrics["cpu"],
        "memory_used": avg_metrics["memory"],
        "swap_used": avg_metrics["swap"],
        "patching": True,
        # Config values (what you passed)
        "slice_width": config["slice_width"],
        "slice_height": config["slice_height"],
        "overlap_ratio": config["overlap_ratio"],
        "x_overlap": x_overlap,
        "y_overlap": y_overlap,
        # Actual values used by SAHI
        "actual_slice_width": actual_slice_width,
        "actual_slice_height": actual_slice_height,
        "actual_num_slices": actual_num_slices,
        "actual_overlap_ratio": 0.2,  # SAHI's default
        "postprocess_type": config["postprocess_type"],
        "postprocess_match_metric": config["postprocess_match_metric"],
        "postprocess_match_threshold": config["postprocess_match_threshold"],
    }

    


In [ ]:
model_path = 'yolo11m'
suffix='sahi'

output_dir = f'demo_output_{model_name}_{suffix}'
input_dir = "bench_reitoria_challenge"
save_figures = 'debug'
run_at_cpu = False


if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Initialize model
model = YOLO(model_path)

if run_at_cpu:
    model = model.to('cpu')

class_names = model.names
num_classes = len(class_names)
target_class = 2
exclude_classes_by_id = [i for i in range(num_classes) if i != target_class]
vehicle_class_ids = {2}

In [ ]:
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=model_path,
    confidence_threshold=0.1,
    device="cpu",
)

In [ ]:
csv_file_path = os.path.join(output_dir, "sahi_default_results.csv")
csv_headers = [
    "image_name",
    "image_width",
    "image_height",
    "actual_slice_width",
    "actual_slice_height",
    "actual_num_slices",
    "actual_overlap_ratio",
    "predicted_cars",
    "processing_time",
    "cpu_usage",
    "memory_used",
    "swap_used",
    "postprocess_type",
    "postprocess_match_metric",
    "postprocess_match_threshold",
    "timestamp",
]

with open(csv_file_path, "w", newline="") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_headers)
    writer.writeheader()


In [ ]:
for input_image in tqdm(os.listdir(input_dir)):
    if input_image.lower().endswith((".jpeg", ".jpg", ".png")):
        img_path = os.path.join(input_dir, input_image)
        img_name = os.path.splitext(input_image)[0]
        image_timestamp = extract_timestamp(input_image, mode="filename_reitoria")
        
        # Get image dimensions
        from sahi.utils.cv import read_image_as_pil
        image_pil = read_image_as_pil(img_path)
        w, h = image_pil.size
        
        # Get SAHI's actual slice calculation
        from sahi.slicing import get_slice_bboxes
        slice_bboxes = get_slice_bboxes(
            image_height=h,
            image_width=w,
            slice_height=None,
            slice_width=None,
            overlap_height_ratio=0.2,
            overlap_width_ratio=0.2,
            auto_slice_resolution=True,
        )
        
        if slice_bboxes:
            first_bbox = slice_bboxes[0]
            actual_slice_width = first_bbox[2] - first_bbox[0]
            actual_slice_height = first_bbox[3] - first_bbox[1]
            actual_num_slices = len(slice_bboxes)
        else:
            actual_slice_width = w
            actual_slice_height = h
            actual_num_slices = 1
        
        print(f"\nImage: {img_name} ({w}x{h})")
        print(f"SAHI calculated: {actual_slice_width}x{actual_slice_height}, {actual_num_slices} slices")
        
        # Run SAHI with default parameters
        start_time = time.time()
        start_metrics = get_system_metrics()
        
        result = get_sliced_prediction(
            img_path,
            detection_model,
            slice_height=None,  # Auto
            slice_width=None,   # Auto
            overlap_height_ratio=0.2,  # Default
            overlap_width_ratio=0.2,   # Default
            postprocess_type="GREEDYNMM",  # Default
            postprocess_match_metric="IOS",  # Default
            postprocess_match_threshold=0.5,  # Default
            verbose=1,  # Set to 1 to see SAHI's output
            auto_slice_resolution=True,
            exclude_classes_by_id=exclude_classes_by_id,
        )
        
        processing_time = time.time() - start_time
        end_metrics = get_system_metrics()
        avg_metrics = {
            "cpu": (start_metrics["cpu"] + end_metrics["cpu"]) / 2,
            "memory": (start_metrics["memory"] + end_metrics["memory"]) / 2,
            "swap": (start_metrics["swap"] + end_metrics["swap"]) / 2,
        }
        
        vehicle_count_after = sum(
            1
            for pred in result.object_prediction_list
            if pred.category.id in vehicle_class_ids
        )
        
        # Save visualization
        if save_figures:
            config_name = f"auto_{actual_slice_width}x{actual_slice_height}_ov0.2_GREEDYNMM_IOS"
            result.export_visuals(
                export_dir=output_dir,
                rect_th=3,
                hide_labels=True,
                hide_conf=True,
                file_name=f"result_SAHI_default_{img_name}",
            )
        
        # Write results
        metrics = {
            "image_name": img_name,
            "image_width": w,
            "image_height": h,
            "actual_slice_width": actual_slice_width,
            "actual_slice_height": actual_slice_height,
            "actual_num_slices": actual_num_slices,
            "actual_overlap_ratio": 0.2,
            "predicted_cars": vehicle_count_after,
            "processing_time": processing_time,
            "cpu_usage": avg_metrics["cpu"],
            "memory_used": avg_metrics["memory"],
            "swap_used": avg_metrics["swap"],
            "postprocess_type": "GREEDYNMM",
            "postprocess_match_metric": "IOS",
            "postprocess_match_threshold": 0.5,
            "timestamp": image_timestamp,
        }
        
        with open(csv_file_path, "a", newline="") as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=csv_headers)
            writer.writerow(metrics)
        
        print(f"Detected: {vehicle_count_after} cars in {processing_time:.2f}s")

In [ ]:
print(csv_file_path)
# Load the CSV file created by your loop
# (Ensure csv_file_path variable is still in memory, or replace with the actual filename string)
df_results = pd.read_csv(csv_file_path)

# 1. See the most recent entries
print("Latest processed images:")
display(df_results.tail())

# 2. Check statistics (e.g., avg processing time, avg cars found)
print("\nStatistics:")
display(df_results[['predicted_cars', 'processing_time', 'cpu_usage']].describe())

# 3. Plot processing time over time to check for spikes
df_results['processing_time'].plot(title="Processing Time per Image", figsize=(10, 4))

In [ ]:
suffix

In [ ]:
model_name

In [ ]:
! python3 compute_metrics.py \
    --model "{model_name}_{suffix}" \
    --parking_metrics "{output_dir}/sahi_default_results.csv" \
    --labels "{labels_csv}" \
    --output_dir "{output_dir}/" \
    --max_spots {15}

In [ ]:
print(f"Contents of {output_dir}:")
print(os.listdir(output_dir))


# output_dir = 'demo_output_yolo11m'
summary_path = os.path.join(output_dir, f'summary_metrics_{model_name}_{suffix}.csv')
# output_path
df = pd.read_csv(summary_path)

# List of values to exclude
exclude = ['Average accuracy', 'Balanced accuracy', 'Average precision', 'Average recall', 'Average F1 score']
df_filtered = df[~df.iloc[:, 0].isin(exclude)]

display(df_filtered)


show_images = 4
# Get all result images
result_images = glob.glob(os.path.join(output_dir, 'result*.png'))

# Display the first image
for img_path in result_images[:show_images]:
    print(f"Displaying: {os.path.basename(img_path)}")
    img = Image.open(img_path)
    display(img)